In [0]:
# Databricks notebook source
# Silver - Squad 3 - ecommerce_clientes
# Fluxo: Bronze Delta -> Silver Delta

In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
BRONZE_TABLE = "ecommerce_clientes"
BRONZE_PATH = f"{BRONZE_BASE_PATH}{BRONZE_TABLE}"

SILVER_TABLE = "ecommerce_clientes"
SILVER_PATH = f"{SILVER_BASE_PATH}{SILVER_TABLE}"

KEY_COLUMNS = ["id_cliente"]

SILVER_WRITE_MODE = "overwrite"

print("BRONZE_PATH:", BRONZE_PATH)
print("SILVER_PATH:", SILVER_PATH)
print("KEY_COLUMNS:", KEY_COLUMNS)

In [0]:
adls_options = get_adls_options()

print("Opções ADLS configuradas.")

In [0]:
df_bronze = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(BRONZE_PATH)
)

df_bronze.printSchema()

total_bronze = df_bronze.count()

print(f"Total de registros na Bronze: {total_bronze}")

display(df_bronze.limit(10))

In [0]:
from pyspark.sql.functions import col, count, when, to_timestamp

df_validacao_conversoes = df_bronze.select(
    count("*").alias("total_linhas"),

    count(
        when(
            col("id_cliente").isNotNull() &
            col("id_cliente").cast("int").isNull(),
            True
        )
    ).alias("falhas_id_cliente"),

    count(
        when(
            col("dt_cadastro").isNotNull() &
            to_timestamp(col("dt_cadastro")).isNull(),
            True
        )
    ).alias("falhas_dt_cadastro"),

    count(
        when(
            col("dt_ultima_atualizacao").isNotNull() &
            to_timestamp(col("dt_ultima_atualizacao")).isNull(),
            True
        )
    ).alias("falhas_dt_ultima_atualizacao")
)

display(df_validacao_conversoes)

validacao_conversoes = df_validacao_conversoes.collect()[0]

if validacao_conversoes["falhas_id_cliente"] > 0:
    raise Exception("Existem valores de id_cliente que não podem ser convertidos para integer.")

if validacao_conversoes["falhas_dt_cadastro"] > 0:
    raise Exception("Existem valores de dt_cadastro que não podem ser convertidos para timestamp.")

if validacao_conversoes["falhas_dt_ultima_atualizacao"] > 0:
    raise Exception("Existem valores de dt_ultima_atualizacao que não podem ser convertidos para timestamp.")

print("Validação OK: conversões principais podem ser feitas.")

In [0]:
from pyspark.sql.functions import (
    col,
    trim,
    lower,
    to_timestamp,
    current_timestamp,
    current_date,
    datediff,
    to_date,
    row_number
)

from pyspark.sql.window import Window

df_silver_base = (
    df_bronze
    .withColumn("id_cliente_int", col("id_cliente").cast("int"))
    .withColumn("dt_cadastro_ts", to_timestamp(col("dt_cadastro")))
    .withColumn("dt_ultima_atualizacao_ts", to_timestamp(col("dt_ultima_atualizacao")))
)

window_deduplicacao = (
    Window
    .partitionBy("id_cliente_int")
    .orderBy(
        col("dt_ultima_atualizacao_ts").desc_nulls_last(),
        col("dt_cadastro_ts").desc_nulls_last(),
        col("bronze_ingested_at").desc_nulls_last()
    )
)

df_silver = (
    df_silver_base
    .withColumn("rn", row_number().over(window_deduplicacao))
    .filter(col("rn") == 1)
    .select(
        col("id_cliente_int").alias("id_cliente"),
        col("uuid_cliente").cast("string").alias("uuid_cliente"),
        trim(col("nome")).alias("nome"),
        trim(col("sobrenome")).alias("sobrenome"),
        lower(trim(col("email"))).alias("email"),
        col("dt_cadastro_ts").alias("dt_cadastro"),
        col("dt_ultima_atualizacao_ts").alias("dt_ultima_atualizacao"),
        datediff(current_date(), to_date(col("dt_cadastro_ts"))).alias("dias_desde_cadastro"),

        col("bronze_ingested_at").cast("timestamp").alias("bronze_ingested_at"),
        col("bronze_source_file").cast("string").alias("bronze_source_file"),

        current_timestamp().alias("silver_processed_at"),

        col("ano").cast("int").alias("ano"),
        col("mes").cast("int").alias("mes")
    )
)

In [0]:
# visualizar DataFrame Silver antes da gravação

df_silver.printSchema()

total_silver = df_silver.count()

print(f"Total de registros na Bronze: {total_bronze}")
print(f"Total de registros na Silver após deduplicação: {total_silver}")

display(df_silver.limit(10))

In [0]:
# validar deduplicação por id_cliente

from pyspark.sql.functions import countDistinct

total_ids_distintos_bronze = (
    df_bronze
    .filter(col("id_cliente").isNotNull())
    .select(col("id_cliente").cast("int").alias("id_cliente"))
    .distinct()
    .count()
)

duplicados_silver = (
    df_silver
    .groupBy("id_cliente")
    .count()
    .filter(col("count") > 1)
    .count()
)

print(f"IDs distintos na Bronze: {total_ids_distintos_bronze}")
print(f"Registros na Silver: {total_silver}")
print(f"IDs duplicados na Silver: {duplicados_silver}")

if total_silver != total_ids_distintos_bronze:
    raise Exception("Erro: quantidade da Silver diferente da quantidade de IDs distintos da Bronze.")

if duplicados_silver > 0:
    raise Exception("Erro: ainda existem id_cliente duplicados na Silver.")

print("Validação OK: Silver deduplicada por id_cliente.")

In [0]:
# validar qualidade dos dados da Silver

from pyspark.sql.functions import count, when

df_validacao_silver = df_silver.select(
    count("*").alias("total_linhas"),
    count(when(col("id_cliente").isNull(), True)).alias("id_cliente_nulo"),
    count(when(col("email").isNull(), True)).alias("email_nulo"),
    count(when(col("dt_cadastro").isNull(), True)).alias("dt_cadastro_nulo"),
    count(when(col("ano").isNull(), True)).alias("ano_nulo"),
    count(when(col("mes").isNull(), True)).alias("mes_nulo"),
    count(when(col("silver_processed_at").isNull(), True)).alias("silver_processed_at_nulo"),
    count(when(col("bronze_ingested_at").isNull(), True)).alias("bronze_ingested_at_nulo"),
    count(when(col("bronze_source_file").isNull(), True)).alias("bronze_source_file_nulo"),
    count(when(col("dias_desde_cadastro").isNull(), True)).alias("dias_desde_cadastro_nulo")
)

display(df_validacao_silver)

validacao_silver = df_validacao_silver.collect()[0]

if validacao_silver["id_cliente_nulo"] > 0:
    raise Exception("Erro: existem registros com id_cliente nulo na Silver.")

if validacao_silver["dt_cadastro_nulo"] > 0:
    raise Exception("Erro: existem registros com dt_cadastro nulo na Silver.")

if validacao_silver["ano_nulo"] > 0:
    raise Exception("Erro: existem registros com ano nulo na Silver.")

if validacao_silver["mes_nulo"] > 0:
    raise Exception("Erro: existem registros com mes nulo na Silver.")

if validacao_silver["silver_processed_at_nulo"] > 0:
    raise Exception("Erro: existem registros sem silver_processed_at.")

if validacao_silver["bronze_source_file_nulo"] > 0:
    raise Exception("Erro: existem registros sem bronze_source_file.")

print("Validação OK: qualidade mínima da Silver aprovada.")

In [0]:
# gravar Silver Delta

(
    df_silver
    .write
    .format("delta")
    .options(**adls_options)
    .option("overwriteSchema", "true")
    .mode(SILVER_WRITE_MODE)
    .partitionBy("ano", "mes")
    .save(SILVER_PATH)
)

print(f"Dados gravados com sucesso na Silver: {SILVER_PATH}")

In [0]:
# ler Silver gravada

df_silver_saved = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(SILVER_PATH)
)

df_silver_saved.printSchema()

total_silver_saved = df_silver_saved.count()

print(f"Total de registros na Silver gravada: {total_silver_saved}")

display(df_silver_saved.limit(10))

In [0]:
# validar quantidade da Silver gravada

print(f"Total Silver em memória: {total_silver}")
print(f"Total Silver gravada: {total_silver_saved}")

if total_silver != total_silver_saved:
    raise Exception("Erro: quantidade da Silver em memória diferente da Silver gravada.")

print("Validação OK: quantidade da Silver gravada confere.")

In [0]:
# validar schema final da Silver

colunas_silver = df_silver_saved.columns

colunas_obrigatorias = [
    "id_cliente",
    "uuid_cliente",
    "nome",
    "sobrenome",
    "email",
    "dt_cadastro",
    "dt_ultima_atualizacao",
    "dias_desde_cadastro",
    "bronze_ingested_at",
    "bronze_source_file",
    "silver_processed_at",
    "ano",
    "mes"
]

colunas_ausentes = [c for c in colunas_obrigatorias if c not in colunas_silver]

if colunas_ausentes:
    raise Exception(f"Erro: colunas obrigatórias ausentes na Silver: {colunas_ausentes}")

if "senha_hash" in colunas_silver:
    raise Exception("Erro: senha_hash não deveria estar na Silver.")

print("Validação OK: schema final da Silver aprovado.")